In [12]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
# PARA MANEJO DE ARCHIVOS XML
import xml.etree.ElementTree as ET

CONEXION A BASE DE DATOS

In [13]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [14]:
# store = pd.read_sql_table("Store",motorBaseDatos,"Sales")

queryStore = """
SELECT
      [Name],
      [BusinessEntityID],
      [SalesPersonID],
      [Demographics]
FROM Sales.Store
"""

tablaStore = pd.read_sql_query(queryStore, motorBaseDatos)



queryGeography = """
SELECT 
[GeographyKey]
FROM dbo.dimensionGeography
"""

dimensionGeography = pd.read_sql_query(queryGeography, motorBodegaDatos)
# dimensionGeography


queryPersonPhone = """
SELECT 
[BusinessEntityID]
      ,[PhoneNumber]
FROM Person.PersonPhone
"""
tablaPersonPhone = pd.read_sql_query(queryPersonPhone, motorBaseDatos)

queryPersonAddress = """
SELECT 
[AddressID]
      ,[AddressLine1]
      ,[AddressLine2]
FROM Person.Address
"""
tablaPersonAddress = pd.read_sql_query(queryPersonAddress, motorBaseDatos)




queryPersonBusinessEntityAddress = """
SELECT 
[BusinessEntityID]
      ,[AddressID]
FROM Person.BusinessEntityAddress
"""
tablaBusinessEntityAddress = pd.read_sql_query(queryPersonBusinessEntityAddress, motorBaseDatos)





queryCustomer = """
SELECT 
      [StoreID],
      [AccountNumber],
      [TerritoryID]
FROM Sales.Customer
"""
tablaCustomer = pd.read_sql_query(queryCustomer, motorBaseDatos)



# tablaStore
# tablaPersonPhone
# tablaPersonAddress
# tablaBusinessEntityAddress

# tablaCustomer


TRANSFORMACION

In [15]:
# Namespace del XML
NS = {'ns': 'http://schemas.microsoft.com/sqlserver/2004/07/adventure-works/StoreSurvey'}

def parse_store_survey(xml_str):
    root = ET.fromstring(xml_str)
    # Extraer cada campo del XML usando el namespace
    return {
        'AnnualSales': root.find('ns:AnnualSales', NS).text if root.find('ns:AnnualSales', NS) is not None else None,
        'AnnualRevenue': root.find('ns:AnnualRevenue', NS).text if root.find('ns:AnnualRevenue', NS) is not None else None,
        'BankName': root.find('ns:BankName', NS).text if root.find('ns:BankName', NS) is not None else None,
        'BusinessType': root.find('ns:BusinessType', NS).text if root.find('ns:BusinessType', NS) is not None else None,
        'YearOpened': root.find('ns:YearOpened', NS).text if root.find('ns:YearOpened', NS) is not None else None,
        'Specialty': root.find('ns:Specialty', NS).text if root.find('ns:Specialty', NS) is not None else None,
        'SquareFeet': root.find('ns:SquareFeet', NS).text if root.find('ns:SquareFeet', NS) is not None else None,
        'Brands': root.find('ns:Brands', NS).text if root.find('ns:Brands', NS) is not None else None,
        'Internet': root.find('ns:Internet', NS).text if root.find('ns:Internet', NS) is not None else None,
        'NumberEmployees': root.find('ns:NumberEmployees', NS).text if root.find('ns:NumberEmployees', NS) is not None else None,
    }

# Supón que df es tu DataFrame y la columna se llama 'Demographics'
# Aplica la función para crear un DataFrame expandido con esos campos
df_expanded = tablaStore['Demographics'].apply(parse_store_survey).apply(pd.Series)

# Combinar con el DataFrame original si lo deseas
store = pd.concat([tablaStore, df_expanded], axis=1)

store

,Name,BusinessEntityID,SalesPersonID,Demographics,AnnualSales,AnnualRevenue,BankName,BusinessType,YearOpened,Specialty,SquareFeet,Brands,Internet,NumberEmployees
0,Next-Door Bike Store,292,279,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,United Security,BM,1996,Mountain,21000,2,ISDN,13
1,Professional Sales and Service,294,276,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,International Bank,BM,1991,Touring,18000,4+,T1,14
2,Riders Company,296,277,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,Primary Bank & Reserve,BM,1999,Road,21000,2,DSL,15
3,The Bike Mechanics,298,275,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,International Security,BM,1994,Mountain,18000,2,DSL,16
4,Nationwide Supply,300,286,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,Guardian Bank,BM,1987,Touring,21000,4+,DSL,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
696,Retreat Inn,1988,282,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",300000,30000,Primary Bank & Reserve,BM,1982,Road,7000,4+,T2,8
697,Technical Parts Manufacturing,1990,281,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",300000,30000,International Security,BM,1976,Touring,7000,4+,T1,5
698,Totes & Baskets Company,1992,277,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",300000,30000,Guardian Bank,BM,1970,Road,6000,4+,DSL,2
699,World of Bikes,1994,277,"<StoreSurvey xmlns=""http://schemas.microsoft.c...",800000,80000,Primary Bank & Reserve,BM,1997,Mountain,19000,4+,T1,17


In [16]:
dimensionReseller = store


dimensionReseller.rename(columns={
    'Name' : 'ResellerName',
    'Specialty' : 'ProductLine',
    'BusinessEntityID': 'StoreID'
}, inplace=True)

dimensionReseller =  dimensionReseller.merge(tablaCustomer, on='StoreID')

dimensionReseller = dimensionReseller.merge(tablaPersonPhone, left_on='SalesPersonID', right_on='BusinessEntityID') 

dimensionReseller.drop(columns=[
    'BusinessEntityID',
    'Demographics',
], inplace=True)



dimensionReseller.drop(columns=[
    'SquareFeet',
    'Brands',
    'Internet',
], inplace=True)

dimensionReseller


,ResellerName,StoreID,SalesPersonID,AnnualSales,AnnualRevenue,BankName,BusinessType,YearOpened,ProductLine,NumberEmployees,AccountNumber,TerritoryID,PhoneNumber
0,Next-Door Bike Store,292,279,800000,80000,United Security,BM,1996,Mountain,13,AW00000585,5,664-555-0112
1,Next-Door Bike Store,292,279,800000,80000,United Security,BM,1996,Mountain,13,AW00029484,5,664-555-0112
2,Professional Sales and Service,294,276,800000,80000,International Bank,BM,1991,Touring,14,AW00000582,4,883-555-0116
3,Professional Sales and Service,294,276,800000,80000,International Bank,BM,1991,Touring,14,AW00029485,4,883-555-0116
4,Riders Company,296,277,800000,80000,Primary Bank & Reserve,BM,1999,Road,15,AW00000579,3,517-555-0117
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,BM,1970,Road,2,AW00000328,4,517-555-0117
1332,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,BM,1970,Road,2,AW00030117,4,517-555-0117
1333,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,BM,1997,Mountain,17,AW00000327,3,517-555-0117
1334,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,BM,1997,Mountain,17,AW00030118,3,517-555-0117


In [17]:
address = tablaBusinessEntityAddress.merge(tablaPersonAddress, on='AddressID')

address.drop(columns=[
    'AddressID',
], inplace=True)

dimensionReseller = dimensionReseller.merge(address, left_on='SalesPersonID', right_on='BusinessEntityID')

dimensionReseller.drop(columns=[
    'BusinessEntityID'
], inplace=True)



dimensionReseller

,ResellerName,StoreID,SalesPersonID,AnnualSales,AnnualRevenue,BankName,BusinessType,YearOpened,ProductLine,NumberEmployees,AccountNumber,TerritoryID,PhoneNumber,AddressLine1,AddressLine2
0,Next-Door Bike Store,292,279,800000,80000,United Security,BM,1996,Mountain,13,AW00000585,5,664-555-0112,8291 Crossbow Way,None
1,Next-Door Bike Store,292,279,800000,80000,United Security,BM,1996,Mountain,13,AW00029484,5,664-555-0112,8291 Crossbow Way,None
2,Professional Sales and Service,294,276,800000,80000,International Bank,BM,1991,Touring,14,AW00000582,4,883-555-0116,2487 Riverside Drive,None
3,Professional Sales and Service,294,276,800000,80000,International Bank,BM,1991,Touring,14,AW00029485,4,883-555-0116,2487 Riverside Drive,None
4,Riders Company,296,277,800000,80000,Primary Bank & Reserve,BM,1999,Road,15,AW00000579,3,517-555-0117,80 Sunview Terrace,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,BM,1970,Road,2,AW00000328,4,517-555-0117,80 Sunview Terrace,None
1332,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,BM,1970,Road,2,AW00030117,4,517-555-0117,80 Sunview Terrace,None
1333,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,BM,1997,Mountain,17,AW00000327,3,517-555-0117,80 Sunview Terrace,None
1334,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,BM,1997,Mountain,17,AW00030118,3,517-555-0117,80 Sunview Terrace,None


In [18]:


dimensionReseller["OrderFrequency"] = None
dimensionReseller["OrderMonth"] = None
dimensionReseller["FirstOrderYear"] = None
dimensionReseller["LastOrderYear"] = None
dimensionReseller["MinPaymentType"] = None
dimensionReseller["MinPaymentAmount"] = None


# DICCIONARIO PARA ORDER-FRECUENCY
diccionarioOrderFrecuency = {
    'OS' : 'Q',
    'BM' : 'S',
    'BS' : 'A'
}

# DICCIONARIO PARA BUSINESS-TYPE
diccionarioBusinessType = {
    'OS' : 'Warehouse',
    'BM' : 'Value Added Reseller',
    'BS' : 'Specialty Bike Shop'
}

dimensionReseller["OrderFrequency"] = dimensionReseller["BusinessType"].replace(diccionarioOrderFrecuency)
dimensionReseller["BusinessType"] = dimensionReseller["BusinessType"].replace(diccionarioBusinessType)
dimensionReseller

,ResellerName,StoreID,SalesPersonID,AnnualSales,AnnualRevenue,BankName,BusinessType,YearOpened,ProductLine,NumberEmployees,...,TerritoryID,PhoneNumber,AddressLine1,AddressLine2,OrderFrequency,OrderMonth,FirstOrderYear,LastOrderYear,MinPaymentType,MinPaymentAmount
0,Next-Door Bike Store,292,279,800000,80000,United Security,Value Added Reseller,1996,Mountain,13,...,5,664-555-0112,8291 Crossbow Way,None,S,None,None,None,None,None
1,Next-Door Bike Store,292,279,800000,80000,United Security,Value Added Reseller,1996,Mountain,13,...,5,664-555-0112,8291 Crossbow Way,None,S,None,None,None,None,None
2,Professional Sales and Service,294,276,800000,80000,International Bank,Value Added Reseller,1991,Touring,14,...,4,883-555-0116,2487 Riverside Drive,None,S,None,None,None,None,None
3,Professional Sales and Service,294,276,800000,80000,International Bank,Value Added Reseller,1991,Touring,14,...,4,883-555-0116,2487 Riverside Drive,None,S,None,None,None,None,None
4,Riders Company,296,277,800000,80000,Primary Bank & Reserve,Value Added Reseller,1999,Road,15,...,3,517-555-0117,80 Sunview Terrace,None,S,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,Value Added Reseller,1970,Road,2,...,4,517-555-0117,80 Sunview Terrace,None,S,None,None,None,None,None
1332,Totes & Baskets Company,1992,277,300000,30000,Guardian Bank,Value Added Reseller,1970,Road,2,...,4,517-555-0117,80 Sunview Terrace,None,S,None,None,None,None,None
1333,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,Value Added Reseller,1997,Mountain,17,...,3,517-555-0117,80 Sunview Terrace,None,S,None,None,None,None,None
1334,World of Bikes,1994,277,800000,80000,Primary Bank & Reserve,Value Added Reseller,1997,Mountain,17,...,3,517-555-0117,80 Sunview Terrace,None,S,None,None,None,None,None


CARGAR A LA BODEGA

In [19]:
dimensionReseller.to_sql('dimensionReseller',motorBodegaDatos, if_exists='replace',index_label='ResellerKey')

6